# Kokborok–English Neural Machine Translation
## Fine-Tuning Meta's M2M100 (418M) on Indigenous Parallel Corpus

**Associated Research Paper:**
*"Developing Neural Machine Translation Tool: Kokborok–English, a Step Toward Preserving Indigenous Languages"*
- **Authors:** Sanchali Das, Pranjal, Mainak Sarkar
- **Published In:** *Sustainable Developments in Computer Engineering, Green Technology and Smart Systems* (CRC Press / Taylor & Francis, 2026)
- **DOI:** [10.1201/9781003743767-121](https://doi.org/10.1201/9781003743767-121)

---
### Overview
This notebook provides the complete fine-tuning pipeline for developing a bilingual Kokborok-to-English Neural Machine Translation system using . Due to the low-resource nature of Kokborok, transfer learning via M2M100's multilingual representation space ( -> ) enables effective convergence even with small corpora.

### Step 1: Install Dependencies
Install Hugging Face , , , and metric dependencies.

In [ ]:
# Disable wandb logging for clean training execution
import os
os.environ["WANDB_DISABLED"] = "true"

!pip install -q transformers datasets evaluate sacrebleu accelerate

### Step 2: Environment and Hardware Verification
Verify CUDA GPU availability (e.g., NVIDIA L4 / A100 on Google Colab Pro).

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Allocated Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

### Step 3: Load Model & Tokenizer
Initialize  with multilingual transfer language tags.

In [ ]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

model_name = "facebook/m2m100_418M"
print(f"Loading tokenizer and pretrained model: {model_name}...")
tokenizer = M2M100Tokenizer.from_pretrained(model_name)
model = M2M100ForConditionalGeneration.from_pretrained(model_name)

# Set source and target language codes for transfer learning
src_lang = "hi"  # Indic transfer language anchor in M2M100
tgt_lang = "en"
tokenizer.src_lang = src_lang
tokenizer.tgt_lang = tgt_lang
print(f"Source language tag: {src_lang} | Target language tag: {tgt_lang}")

### Step 4: Dataset Preparation and Subword Tokenization
Load the parallel corpus, map tokenization with sequence truncation and padding to 64 tokens.

In [ ]:
import pandas as pd
from datasets import Dataset as HFDataset

# Specify path to Kokborok-English parallel CSV dataset
DATASET_PATH = "kokborok_full_sentences.csv"

# Read parallel corpus (columns: "kokborok", "english")
df = pd.read_csv(DATASET_PATH)
df.columns = [c.strip().lower() for c in df.columns]
df = df.dropna(subset=["kokborok", "english"])
print(f"Total parallel sentence pairs loaded: {len(df):,}")

src_texts = df["kokborok"].astype(str).tolist()
tgt_texts = df["english"].astype(str).tolist()

# Subword Tokenization (Max sequence length = 64)
def preprocess_batch(batch):
    return tokenizer(
        batch["src_text"],
        text_target=batch["tgt_text"],
        max_length=64,
        padding="max_length",
        truncation=True
    )

hf_dataset = HFDataset.from_dict({"src_text": src_texts, "tgt_text": tgt_texts})
tokenized_dataset = hf_dataset.map(preprocess_batch, batched=True, remove_columns=["src_text", "tgt_text"])
print(f"Tokenized dataset ready with {len(tokenized_dataset):,} samples.")

### Step 5: Training Configuration (Hyperparameters Verified from Paper)
Configure Hugging Face  matching the published research paper:
- **Optimizer:** AdamW
- **Learning Rate:** 
- **Batch Size:**  per device
- **Epochs:** 
- **Weight Decay:** 
- **Mixed Precision (FP16):** Enabled on GPU
- **Logging Steps:** 

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./m2m100_finetuned",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    num_train_epochs=12,
    weight_decay=0.01,
    logging_steps=500,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)
print("Trainer initialized successfully.")

### Step 6: Model Training
Execute the fine-tuning loop over 12 epochs.

In [ ]:
import time

start_time = time.time()
trainer.train()
elapsed = time.time() - start_time
print(f"Training finished in {elapsed:.1f} seconds ({elapsed / 3600:.2f} hours).")

### Step 7: Save Checkpoint and Tokenizer
Persist fine-tuned weights and tokenizer for inference and downstream evaluation.

In [ ]:
save_directory = "./m2m100_final"
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"Final fine-tuned model saved to {save_directory}")